In [3]:
from agents import PyTorchTronAgent, TronBattleRunner
import torch
from agent_classes import DeepQAgent
from agents import watch_policy
from controller import GreedySpaceController

In [4]:
# Load your saved PyTorch models (they can be from any framework)
agent = DeepQAgent()
agent.load('tron_genomes/deepq_model.pth')

agent2 = DeepQAgent()
agent2.load('tron_genomes/deepq_model.pth')

runner = TronBattleRunner(width=32, height=32, envs=256)

result = runner.battle([agent, agent2], episodes=100)
print(f"Agent A wins: {result.wins[0]}, Agent B wins: {result.wins[1]}, Draws: {result.draws}")

Model loaded from tron_genomes/deepq_model.pth
Model loaded from tron_genomes/deepq_model.pth
Agent A wins: 1, Agent B wins: 0, Draws: 99


In [5]:
# TwoAgentPolicy adapter (place in a cell in src/agentNotebook.ipynb)
import numpy as np

class TwoAgentPolicy:
    def __init__(self, agent_a, agent_b):
        self.agents = [agent_a, agent_b]

    def actions(self, model):
        # expects model.obs shape (envs, players, ... ) or agents that accept model
        obs = getattr(model, "obs", None)
        envs = obs.shape[0] if obs is not None else None
        acts = []

        for p, agent in enumerate(self.agents):
            # try several common agent APIs
            try:
                a = agent.actions(model)                      # common API
            except TypeError:
                try:
                    a = agent.actions(model, p)               # some agents accept player idx
                except Exception:
                    if obs is None:
                        raise RuntimeError("Agent requires per-player obs but model has no .obs")
                    # assume agent.act takes per-player observations with shape (envs, ...)
                    a = agent.act(obs[:, p])
            acts.append(np.asarray(a).reshape(envs, -1)[:, 0] if envs is not None else np.asarray(a))

        # return shape (envs, players) as expected by model.step(...)
        return np.stack(acts, axis=1)

In [6]:
# Usage examples (place in a separate cell in src/agentNotebook.ipynb)
from agents import watch_policy

# Example A: watch single agent self-play (if agent implements actions(model))
watch_policy(agent, players=2, width=32, height=32, scale=12, fps=15, seed=0)

# Example B: watch two different agents
policy = TwoAgentPolicy(agent, agent2)
watch_policy(policy, players=2, width=32, height=32, scale=12, fps=15, seed=0)

AttributeError: 'DeepQAgent' object has no attribute 'actions'